In [1]:
""" Import required libraries
"""
import torch
import scipy.io as sio
import numpy as np
import netCDF4 as nc
import deepxde as dde
import pinnicle as pinn
# import matplotlib.tri as tri
# import matplotlib.pyplot as plt
# from matplotlib.patches import Polygon
# from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, LogNorm
# from matplotlib.artist import Artist
from matplotlib.path import Path
# from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm
dde.config.set_default_float('float32')
# import cartopy.crs as ccrs
# import matplotlib.ticker as mticker
# from pyproj import Transformer
# from matplotlib.lines import Line2D
# from matplotlib.legend_handler import HandlerBase
# from matplotlib.image import BboxImage
# from matplotlib.transforms import Bbox
# import matplotlib.colors as mcolors

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Set the default float type to float64
Set the default float type to float32


In [2]:
cd /private/var/phys-admin/Documents/Jakob/MC-SSA

/private/var/phys-admin/Documents/Jakob/MC-SSA


/opt/miniconda3/envs/ddeenv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
""" specify data paths
"""
glacier = "Byrd-BMap3GL"
size = "30" + "ma"

# FFT = False

# model = "MC"
# model = "MCexact"
model = "MCexactHelm"

homepath = '/private/var/phys-admin/Documents/Jakob/MC-SSA'
datapath = homepath + '/data/'
figurepath = homepath + '/figures/'
folder = homepath + '/MC-SSA/domains/'
savepath = homepath + '/saved-data/'

# for Gadi runs
outline = folder + f"{glacier}-{size}-vertices.exp"
H_radar = folder + 'bedmap3_radar_thk_profiles.mat'
# dHdt_il = folder + 'ItsLIVE_dHdt_2013-18_mps.mat'
# smb_noell = folder + 'Noell_RACMO_smb_2013-17_mps.mat'
app_mb = folder + 'apparent_mass_balance_2013-17_mps.mat'

bm3 = datapath + 'measures_data/BedMachineAntarctica-v3.nc'
bmap3 = datapath + '/BedMap3-radar-thk-profiles/bedmap3.nc'
meas = datapath + 'measures_data/antarctic_ice_vel_phase_map_v01_seconds.nc'
itslive = datapath + 'itslive_data/ITS_LIVE_velocity_120m_RGI19A_0000_v02.nc'

outFile = glacier
modelFolder = 'MC-SSA/Models/' + outFile 

In [4]:
""" create the PINNICLE parameter dictionary
"""
hp = {}
# hp['epochs'] = 1000
# hp['learning_rate'] = 1e-3
# hp['loss_function'] = 'MSE'
hp['save_path'] = modelFolder
# hp['is_save'] = False
# hp['is_plot'] = True

# hp["fft"] = FFT
# hp["sigma"] = [1.0, 10.0, 100.0]
# hp["num_fourier_feature"] = 50

hp["activation"] = "SiLU" 
hp["initializer"] = "Glorot uniform"
hp["num_neurons"] = 50
hp["num_layers"] = 8
# hp["optimizer"] = "L-BFGS"

hp['shapefile'] = outline
# hp['num_collocation_points'] = 5000

MC = {}
MC_exact = {}
MC_exact_Helm = {}
if model == 'MC':
    hp["equations"] = {"MC":MC}
elif model == "MCexact":
    hp["equations"] = {"MC_exact":MC_exact}
elif model == "MCexactHelm":
    hp["equations"] = {"MC_exact_Helmholtz":MC_exact_Helm}

hp ['data'] = {}

In [5]:
""" load the trained neural networks
    five per experiment
"""
expNo = 1
date = '24-06-26'
specifier = '' # '-attempt4' #
experiments = [None]*5
for k in range(5):
    experiments[k] = pinn.PINN(hp)
    experiments[k].load_model(path=f'{homepath}/MC-SSA/Models/{glacier}', 
                        subfolder='pinn', 
                        name=f'Byrd-BMap3GL-30ma-MCexactHelm-exp{expNo}{specifier}-No{k+1}-{date}',
                        epochs=700000,
                        fileformat='pt')

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-1000. -1000.    -2.] and [1000.  1000.     8.5]
Compiling model...
'compile' took 0.000099 s

Compiling model...
'compile' took 0.000235 s

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-1000. -1000.    -2.] and [1000.  1000.     8.5]
Compiling model...
'compile' took 0.000058 s

Compiling model...
'compile' took 0.000197 s

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 647310.2500, -645047.0625], device='mps:0')
add output transform with [-1000. -1000.    -2.] and [1000.  1000.     8.5]
Compiling model...
'compile' took 0.000047 s

Compiling model...
'compile' took 0.000191 s

add input transform with tensor([ 375692.6875, -983606.9375], device='mps:0') and tensor([ 

In [6]:
""" load the required data
"""
ds_vel = nc.Dataset(meas)
ds_bmac = nc.Dataset(bm3)
ds_bmap = nc.Dataset(bmap3)
ds_il = nc.Dataset(itslive)

# ds_H = sio.loadmat(H_radar)
ds_mb = sio.loadmat(app_mb)

In [13]:
""" create the relevant coordinate arrays and grids 
"""
# load data

x_v = ds_vel['x'][:]
y_v = ds_vel['y'][:]
x_v2 = ds_il['x'][:]
y_v2 = ds_il['y'][:]
x_H1 = ds_bmac['x'][:]
y_H1 = ds_bmac['y'][:]
x_H2 = ds_bmap['x'][:]
y_H2 = ds_bmap['y'][:]
# x_H = ds_H['x'][:]
# y_H = ds_H['y'][:]
x_mb = ds_mb['x'][:]
y_mb = ds_mb['y'][:]

# # BM3, Measures
# vx = ds_vel['VX']
# vy = ds_vel['VY']
# s = ds_bmap['surface_topography']
# b = ds_geom['thickness']
# H = ds_H['ice_thickness']
# mb = ds_mb['a-2013-17']
# bed = ds_geom['bed']
# bed2 = ds_bmap['bed_topography']
# s2 = ds_geom['surface']

# # Rema, ItsLive
# vx = ds_vel['vx']
# vy = ds_vel['vy']
# s = ds_geom['__xarray_dataarray_variable__']
# H = ds_H['H']

# thickness
# b = ds_geom['thickness']
# b2 = ds_bmap['ice_thickness']
# b2flmask = np.where(ds_bmap['thickness_uncertainty'][:] < 13., 1, 0)
# b2fl = np.where(b2flmask,b2,np.nan)

vertices = np.loadtxt(outline,delimiter=' ',skiprows=5)
minx = np.min(vertices[:,0])
maxx = np.max(vertices[:,0])
miny = np.min(vertices[:,1])
maxy = np.max(vertices[:,1])
minmax = [minx, maxx, miny, maxy]


# create masks

xvmask = np.where(np.logical_and(x_v>(minx),x_v<(maxx)))[0]
yvmask = np.where(np.logical_and(y_v>(miny),y_v<(maxy)))[0]

xv2mask = np.where(np.logical_and(x_v2>(minx),x_v2<(maxx)))[0]
yv2mask = np.where(np.logical_and(y_v2>(miny),y_v2<(maxy)))[0]

xH1mask = np.where(np.logical_and(x_H1>(minx),x_H1<(maxx)))[0]
yH1mask = np.where(np.logical_and(y_H1>(miny),y_H1<(maxy)))[0]

xH2mask = np.where(np.logical_and(x_H2>(minx),x_H2<(maxx)))[0]
yH2mask = np.where(np.logical_and(y_H2>(miny),y_H2<(maxy)))[0]


# Hmask = np.where(np.logical_and(
#     np.logical_and(x_H>(minx),x_H<(maxx)),
#     np.logical_and(y_H>(miny),y_H<(maxy))
#     ))

mbmask = np.where(np.logical_and(
    np.logical_and(x_mb>(minx),x_mb<(maxx)),
    np.logical_and(y_mb>(miny),y_mb<(maxy))
    ))



# crop data

# vxcrop = vx[yvmask,:]
# vxcrop = vxcrop[:,xvmask]
# vycrop = vy[yvmask,:]
# vycrop = vycrop[:,xvmask]

# scrop = s[yb2mask,:]
# scrop = scrop[:,xb2mask]

# s2crop = s2[ysmask,:]
# s2crop = s2crop[:,xsmask]

# bedcrop = bed[ysmask,:]
# bedcrop = bedcrop[:,xsmask]

# bed2crop = bed2[yb2mask,:]
# bed2crop = bed2crop[:,xb2mask]

# bcrop = b[ysmask,:]
# bcrop = bcrop[:,xsmask]

# b2crop = b2[yb2mask,:]
# b2crop = b2crop[:,xb2mask]

# b2flcrop = b2fl[yb2mask,:]
# b2flcrop = b2flcrop[:,xb2mask]

# H_crop = H[Hmask]
# mb_crop = mb[mbmask]


# grid data

Xv,Yv = np.meshgrid(x_v[xvmask],y_v[yvmask])
Xvf = Xv.flatten()
Yvf = Yv.flatten()
Xvf = Xvf.reshape((Xvf.size,1))
Yvf = Yvf.reshape((Yvf.size,1))
XYv = np.hstack((Xvf,Yvf))

Xv2,Yv2 = np.meshgrid(x_v2[xv2mask],y_v2[yv2mask])
Xv2f = Xv2.flatten()
Yv2f = Yv2.flatten()
Xv2f = Xv2f.reshape((Xv2f.size,1))
Yv2f = Yv2f.reshape((Yv2f.size,1))
XYv2 = np.hstack((Xv2f,Yv2f))

XH1,YH1 = np.meshgrid(x_H1[xH1mask],y_H1[yH1mask])
XH1f = XH1.flatten()
YH1f = YH1.flatten()
XH1f = XH1f.reshape((XH1f.size,1))
YH1f = YH1f.reshape((YH1f.size,1))
XYH1 = np.hstack((XH1f,YH1f))

XH2,YH2 = np.meshgrid(x_H2[xH2mask],y_H2[yH2mask])
XH2f = XH2.flatten()
YH2f = YH2.flatten()
XH2f = XH2f.reshape((XH2f.size,1))
YH2f = YH2f.reshape((YH2f.size,1))
XYH2 = np.hstack((XH2f,YH2f))

# XH = x_H[Hmask]
# YH = y_H[Hmask]
# XH = XH.reshape((XH.size,1))
# YH = YH.reshape((YH.size,1))
# XYH = np.hstack((XH,YH))

Xmb = x_mb[mbmask]
Ymb = y_mb[mbmask]
Xmb = Xmb.reshape((Xmb.size,1))
Ymb = Ymb.reshape((Ymb.size,1))
XYmb = np.hstack((Xmb,Ymb))

In [14]:
""" define the operators for the network prediction 
"""
experiment = experiments[0]

def OP_mf_mag(i,o):
    return experiment.physics.mf_mag(i,o,None)

def OP_mb_MC(i,o):
    return experiment.physics.smb_MC(i,o,None)

def OP_H(i,o):
    return experiment.physics.get_H(i,o)

def OP_vel_mag_MC(i,o):
    return experiment.physics.vel_mag_MC(i,o,None)

def OP_s(i,o):
    return experiment.physics.get_s(i,o)

def OP_u_MC(i,o):
    return experiment.physics.u_MC(i,o,None)

def OP_v_MC(i,o):
    return experiment.physics.v_MC(i,o,None)

def OP_mf_rot(i,o):
    _, _, _, _, Rx, Ry = experiment.physics.DR_xy(i,o)
    return torch.sqrt(Ry**2 + Rx**2)

def OP_mf_div(i,o):
    Dxmb, Dymb, _, _, _, _ = experiment.physics.DR_xy(i,o)
    return torch.sqrt(Dxmb**2 + Dymb**2)

def OP_res_MC(i,o):
    return experiment.physics.pdes(i,o)[0]

In [15]:
""" evaluate the network prediction 
"""
# create empty arrays

D = np.zeros((5,XYv.shape[0]))
R = np.zeros((5,XYv.shape[0]))

vel_p = np.zeros((5,XYv.shape[0]))
mf_p  = np.zeros((5,XYv.shape[0]))
mf_rot = np.zeros((5,XYv.shape[0]))
mf_div = np.zeros((5,XYv.shape[0]))
pde_res = np.zeros((5,XYv.shape[0]))
mb_p = np.zeros((5,XYmb.shape[0]))
# H_p = np.zeros((5,XYH.shape[0]))
H_pfull = np.zeros((5,XYH1.shape[0]))
H_pb = np.zeros((5,XYH1.shape[0]))
H_pb2 = np.zeros((5,XYH2.shape[0]))
H_il = np.zeros((5,XYv2.shape[0]))

# define step sizes for the loop

step = 50

XYv_step = int(XYv.shape[0]/step)
XYv_rest = int(XYv.shape[0]) - step*XYv_step

XYv2_step = int(XYv2.shape[0]/step)
XYv2_rest = int(XYv2.shape[0]) - step*XYv2_step

XYmb_step = int(XYmb.shape[0]/step)
XYmb_rest = int(XYmb.shape[0]) - step*XYmb_step

# XYH_step = int(XYH.shape[0]/step)
# XYH_rest = int(XYH.shape[0]) - step*XYH_step

XYH1_step = int(XYH1.shape[0]/step)
XYH1_rest = int(XYH1.shape[0]) - step*XYH1_step

XYH2_step = int(XYH2.shape[0]/step)
XYH2_rest = int(XYH2.shape[0]) - step*XYH2_step

# evaluate the network
for k in np.arange(5):
    for i in tqdm(np.arange(step)):
        D[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,0]
        R[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step])[:,1]

        vel_p[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_vel_mag_MC)[:,0]

        mf_p[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_mag)[:,0]

        mf_rot[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_rot)[:,0]
        mf_div[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_mf_div)[:,0]
        
        mb_p[k,i*XYmb_step:(i+1)*XYmb_step] = experiment.model.predict(XYmb[i*XYmb_step:(i+1)*XYmb_step],operator=OP_mb_MC)[:,0]

        # H_p[k,i*XYH_step:(i+1)*XYH_step] = experiment.model.predict(XYH[i*XYH_step:(i+1)*XYH_step],operator=OP_H)[:,0]
        H_pfull[k,i*XYH1_step:(i+1)*XYH1_step] = experiment.model.predict(XYH1[i*XYH1_step:(i+1)*XYH1_step],operator=OP_H)[:,0]

        H_pb[k,i*XYH1_step:(i+1)*XYH1_step] = experiment.model.predict(XYH1[i*XYH1_step:(i+1)*XYH1_step],operator=OP_H)[:,0]
        H_pb2[k,i*XYH2_step:(i+1)*XYH2_step] = experiment.model.predict(XYH2[i*XYH2_step:(i+1)*XYH2_step],operator=OP_H)[:,0]
        H_il[k,i*XYv2_step:(i+1)*XYv2_step] = experiment.model.predict(XYv2[i*XYv2_step:(i+1)*XYv2_step],operator=OP_H)[:,0]

        if model == 'MC':
            pde_res[k,i*XYv_step:(i+1)*XYv_step] = experiment.model.predict(XYv[i*XYv_step:(i+1)*XYv_step],operator=OP_res_MC)[:,0]


    if not XYv_rest==0:
        D[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:])[:,0]
        R[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:])[:,1]

        vel_p[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:],operator=OP_vel_mag_MC)[:,0]

        mf_p[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:],operator=OP_mf_mag)[:,0]

        mf_rot[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:],operator=OP_mf_rot)[:,0]
        mf_div[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:],operator=OP_mf_div)[:,0]
        
        if model == 'MC':
            pde_res[k,-XYv_rest:] = experiment.model.predict(XYv[-XYv_rest:],operator=OP_res_MC)[:,0]

    if not XYmb_rest==0:
        mb_p[k,-XYmb_rest:] = experiment.model.predict(XYmb[-XYmb_rest:],operator=OP_mb_MC)[:,0]

    # if not XYH_rest==0:
    #     H_p[k,-XYH_rest:] = experiment.model.predict(XYH[-XYH_rest:],operator=OP_H)[:,0]

    if not XYH1_rest==0:
        H_pfull[k,-XYH1_rest:] = experiment.model.predict(XYH1[-XYH1_rest:],operator=OP_H)[:,0]
        H_pb[k,-XYH1_rest:] = experiment.model.predict(XYH1[-XYH1_rest:],operator=OP_H)[:,0]

    if not XYH2_rest==0:
        H_pb2[k,-XYH2_rest:] = experiment.model.predict(XYH2[-XYH2_rest:],operator=OP_H)[:,0]

    if not XYv2_rest==0:
        H_il[k,-XYv2_rest:] = experiment.model.predict(XYv2[-XYv2_rest:],operator=OP_H)[:,0]

100%|██████████| 50/50 [00:01<00:00, 25.64it/s]


In [17]:
## create masks
poly_path = Path(vertices)
inside_v = poly_path.contains_points(XYv)
inside_H1 = poly_path.contains_points(XYH1)
inside_H2 = poly_path.contains_points(XYH2)
inside_Hil = poly_path.contains_points(XYv2)
inside_mb = poly_path.contains_points(XYmb)

In [ ]:
## D, R, mass flux (Measures grid)

# normalise D,R gauges s.t. min(R)==min(D)==0
Rmean = np.nanmean(np.where(inside_v,R,np.nan),axis=1)
Dmean = np.nanmean(np.where(inside_v,D,np.nan),axis=1)
Rnorm = Dnorm = np.zeros(np.shape(R))
for i in np.arange(5):
    Rnorm[i,:] = np.where(inside_v,R[i,:] - Rmean[i],np.nan)
    Dnorm[i,:] = np.where(inside_v,D[i,:] - Dmean[i],np.nan)
    Rnorm[i,:] = Rnorm[i,:] - np.nanmin(Rnorm[i,:])
    Dnorm[i,:] = Dnorm[i,:] - np.nanmin(Dnorm[i,:])

meanD = np.mean(np.where(inside_v,Dnorm,np.nan),axis=0)
meanR = np.mean(np.where(inside_v,Rnorm,np.nan),axis=0)
meanMF = np.mean(np.where(inside_v,mf_p,np.nan),axis=0)
meanMFdiv = np.mean(np.where(inside_v,mf_div,np.nan),axis=0)
meanMFrot = np.mean(np.where(inside_v,mf_rot,np.nan),axis=0)

data = {"x":Xv,"y":Yv,"mean_D":meanD,"mean_R":meanR,"mean_MF":meanMF,"mean_MF_div":meanMFdiv,"mean_MF_rot":meanMFrot}
sio.savemat(f'{savepath}exp{expNo}-potentials-and-mass-flux-Measures-grid.mat',data)

In [ ]:
## D, R, mass flux (Measures grid)
D = np.where(inside_v,D[0,:],np.nan)
R = np.where(inside_v,R[0,:],np.nan)
MF = np.where(inside_v,mf_p[i,:],np.nan)
MFdiv = np.where(inside_v,mf_div[i,:],np.nan)
MFrot = np.where(inside_v,mf_rot[i,:],np.nan)

data = {"x":Xv,"y":Yv,"D":D,"R":R,"MF":MF,"MF_div":MFdiv,"MF_rot":MFrot}
sio.savemat(f'{savepath}exp{expNo}-No1-potentials-and-mass-flux-Measures-grid.mat',data)

In [ ]:
## velocity (Measures grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
vel_p = vel_p*spy

mean = np.mean(np.where(inside_v,vel_p,np.nan),axis=0)
median = np.median(np.where(inside_v,vel_p,np.nan),axis=0)
variance = np.var(np.where(inside_v,vel_p,np.nan),axis=0)
mini = np.min(np.where(inside_v,vel_p,np.nan),axis=0)
maxi = np.max(np.where(inside_v,vel_p,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv,"y":Yv,"mean_velocity":mean,"median_velocity":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-surface-velocity-Measures-grid.mat',data)

In [ ]:
## ice thickness (BedMachine grid)
mean = np.mean(np.where(inside_H1,H_pfull,np.nan),axis=0)
median = np.median(np.where(inside_H1,H_pfull,np.nan),axis=0)
variance = np.var(np.where(inside_H1,H_pfull,np.nan),axis=0)
mini = np.min(np.where(inside_H1,H_pfull,np.nan),axis=0)
maxi = np.max(np.where(inside_H1,H_pfull,np.nan),axis=0)
range = maxi-mini

data = {"x":XH1,"y":YH1,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-BedMachine-grid.mat',data)

In [ ]:
## ice thickness (BedMap grid)
mean = np.mean(np.where(inside_H2,H_pb2,np.nan),axis=0)
median = np.median(np.where(inside_H2,H_pb2,np.nan),axis=0)
variance = np.var(np.where(inside_H2,H_pb2,np.nan),axis=0)
mini = np.min(np.where(inside_H2,H_pb2,np.nan),axis=0)
maxi = np.max(np.where(inside_H2,H_pb2,np.nan),axis=0)
range = maxi-mini

data = {"x":XH2,"y":YH2,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-BedMap-grid.mat',data)

In [ ]:
## ice thickness (ItsLive grid)
mean = np.mean(np.where(inside_Hil,H_il,np.nan),axis=0)
median = np.median(np.where(inside_Hil,H_il,np.nan),axis=0)
variance = np.var(np.where(inside_Hil,H_il,np.nan),axis=0)
mini = np.min(np.where(inside_Hil,H_il,np.nan),axis=0)
maxi = np.max(np.where(inside_Hil,H_il,np.nan),axis=0)
range = maxi-mini

data = {"x":Xv2,"y":Yv2,"mean_thickness":mean,"median_thickness":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-ice-thickness-ItsLive-grid.mat',data)

In [ ]:
## apparent mass balance (BedMap grid)

# convert from m/s to m/a
spy = 31536000.  # seconds per year
mb_p = mb_p*spy


mean = np.mean(np.where(inside_mb,mb_p,np.nan),axis=0)
median = np.median(np.where(inside_mb,mb_p,np.nan),axis=0)
variance = np.var(np.where(inside_mb,mb_p,np.nan),axis=0)
mini = np.min(np.where(inside_mb,mb_p,np.nan),axis=0)
maxi = np.max(np.where(inside_mb,mb_p,np.nan),axis=0)
range = maxi-mini

xg = np.unique(Xmb)
yg = np.unique(Ymb)
meanR = np.full((len(yg),len(xg)),np.nan)
medianR = np.full((len(yg),len(xg)),np.nan)
varianceR = np.full((len(yg),len(xg)),np.nan)
miniR = np.full((len(yg),len(xg)),np.nan)
maxiR = np.full((len(yg),len(xg)),np.nan)
rangeR = np.full((len(yg),len(xg)),np.nan)

ix = np.searchsorted(xg,Xmb)
iy = np.searchsorted(yg,Ymb)

meanR[iy,ix] = mean.reshape([len(mean),1])
medianR[iy,ix] = median.reshape([len(mean),1])
varianceR[iy,ix] = variance.reshape([len(mean),1])
miniR[iy,ix] = mini.reshape([len(mean),1])
maxiR[iy,ix] = maxi.reshape([len(mean),1])
rangeR[iy,ix] = range.reshape([len(mean),1])

data = {"x":Xmb,"y":Ymb,"mean_app_mb":mean,"median_app_mb":median,"variance":variance,"min_max_range":range}
sio.savemat(f'{savepath}exp{expNo}-apparent-mass-balance-BedMap-grid.mat',data)